## Working with data in Pandas and SQLite3

In [9]:
import csv
import pandas as pd
import sqlite3 as sql

from pathlib import Path
from src.backend.tools import *
from src.backend.get_pars import *

In [ ]:
# getting the project root
project_root = Path.cwd()
while (project_root.name != "backend") and (project_root.parent != project_root):
    project_root = project_root.parent

csv_dir = project_root / "data/csv"

df_min = 1990
df_max = 2026

df = pd.read_csv(csv_dir/f"pars{df_min}.csv")
for file in csv_dir.glob("*.csv"):

    # only get data in a certain year range
    if df_min < int(file.stem[-4:]) <= df_max :
        _df = pd.read_csv(file)
        df = pd.concat([df, _df], ignore_index=True)

df.to_csv(project_root/"data/par_yields_all.csv", index=False)

In [8]:
csv_file = "./data/par_yields_all.csv"
db_file = "./data/par_yields_all.db"

conn = sql.connect(db_file)
df.to_sql("par_yields_all", conn, index=False)
conn.close()

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9180 entries, 0 to 9179
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    9180 non-null   str    
 1   1M      6280 non-null   float64
 2   2M      1975 non-null   float64
 3   3M      9176 non-null   float64
 4   4M      973 non-null    float64
 5   6M      9179 non-null   float64
 6   1Y      9179 non-null   float64
 7   2Y      9179 non-null   float64
 8   3Y      9179 non-null   float64
 9   5Y      9179 non-null   float64
 10  7Y      9179 non-null   float64
 11  10Y     9179 non-null   float64
 12  20Y     8240 non-null   float64
 13  30Y     8185 non-null   float64
dtypes: float64(13), str(1)
memory usage: 1.1 MB


In [41]:
df.loc[df["1Y"].isnull()]

,date,1M,2M,3M,4M,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
6122,2010-10-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
### Using `sqlite3` directly

In [30]:
# list of maturities we want
months = [1,2,3,4,6];
years = [1,2,3,5,7,10,20,30];
MATURITIES = [ (i/12, f"{i}MONTH") for i in months ]
MATURITIES.extend([ (1.0*i, f"{i}YEAR") for i in years ])
TAUS = [x[0] for x in MATURITIES]

def lookup_par_date(date:str):

    conn = sql.connect("./data/par_yields_all.db")
    cur = conn.cursor()

    cur.execute(f"SELECT * FROM par_yields_all WHERE \"date\" = \"{date}\";")

    fetched = cur.fetchall()

    cur.close()
    conn.close()

    if fetched == [] : return {date: []}
    else:
        _out = {}
        for row in fetched :
            _date = row[0]
            ser = row[1:]
            _out[_date] = list(zip(TAUS, ser))

    return _out

lookup_par_date("2024-11-21")

{'2024-11-21': [(0.08333333333333333, 4.72),
  (0.16666666666666666, 4.67),
  (0.25, 4.63),
  (0.3333333333333333, 4.52),
  (0.5, 4.45),
  (1.0, 4.39),
  (2.0, 4.34),
  (3.0, 4.3),
  (5.0, 4.3),
  (7.0, 4.36),
  (10.0, 4.43),
  (20.0, 4.68),
  (30.0, 4.61)]}

In [ ]:
import sqlite3

def main():
    conn = sqlite3.connect("db01")
    cur = conn.cursor()

    # note: cur.execute is more like an 'initialized iterator' for your query. It doesn't actually pull the data
    # until you do something like `cur.fetchall()` or `cur.fetchone()` or `cur.fetchmany()` (??).
    # this prevents something like trying to pull in a list of millions of entries all at once.
    # cur.execute("SELECT * FROM pars202411 WHERE age > ?", (18,))
    cur.execute("SELECT * FROM pars202411")

    row = cur.fetchone()
    print(type(row))
    print(type(row[0]), "\t", type(row[1]))

    cur.close()
    conn.close()